# Feature Engineering
Feature engineering is the process of using domain knowledge to create new features from existing data. It involves transforming raw data into a format that is more suitable for machine learning algorithms, which can lead to improved model performance.
- In this notebook of the house price predictor project, I will be performing feature engineering on the dataset to create new features that can help improve the predictive power of the model. This may include creating interaction terms, polynomial features, or aggregating existing features to capture more complex relationships in the data.

##1. Import Packages & Environment Setup

In [267]:
%pip install catboost

In [268]:
import os
import sys
import pickle
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from google.colab import userdata

from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor, GradientBoostingRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

##2. Data Ingestion

In [269]:
# 1. Load credentials from google colab credentials

DB_USER = userdata.get("SUPABASE_USER")
DB_PASSWORD = quote_plus(userdata.get("SUPABASE_PASSWORD"))
DB_HOST = userdata.get("SUPABASE_HOST")
DB_PORT = userdata.get("SUPABASE_PORT")
DB_NAME = userdata.get("SUPABASE_NAME")

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

try:
    engine = create_engine(DATABASE_URL)
    with engine.connect() as conn:
        df = pd.read_sql(text("SELECT * FROM housing_data"), conn)
    print("Data loaded successfully from Supabase PostgreSQL database.")
except Exception as e:
    print(f"PostgreSQL connection failed ({e}). Loading from raw CSV fallback...")
    raw_path = os.path.join("..", "data", "raw", "AmesHousing.csv")
    if not os.path.exists(raw_path):
        raw_path = os.path.join("data", "raw", "AmesHousing.csv")
    df = pd.read_csv(raw_path)
    print(f"Data loaded successfully from local CSV: {raw_path}")

# Drop PID / Order columns
drop_cols = [c for c in ["Order", "PID", "Id"] if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)

print(f"Dataset Shape: {df.shape}")
df.head()

Data loaded successfully from Supabase PostgreSQL database.
Dataset Shape: (2930, 80)


,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition,SalePrice
0,20,RL,141.0,31770,Pave,None,IR1,Lvl,AllPub,Corner,...,0,None,None,None,0,5,2010,WD,Normal,215000
1,20,RH,80.0,11622,Pave,None,Reg,Lvl,AllPub,Inside,...,0,None,MnPrv,None,0,6,2010,WD,Normal,105000
2,20,RL,81.0,14267,Pave,None,IR1,Lvl,AllPub,Corner,...,0,None,None,Gar2,12500,6,2010,WD,Normal,172000
3,20,RL,93.0,11160,Pave,None,Reg,Lvl,AllPub,Corner,...,0,None,None,None,0,4,2010,WD,Normal,244000
4,60,RL,74.0,13830,Pave,None,IR1,Lvl,AllPub,Inside,...,0,None,MnPrv,None,0,3,2010,WD,Normal,189900


In [270]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 80 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   MS_SubClass      2930 non-null   int64  
 1   MS_Zoning        2930 non-null   object 
 2   Lot_Frontage     2440 non-null   float64
 3   Lot_Area         2930 non-null   int64  
 4   Street           2930 non-null   object 
 5   Alley            198 non-null    object 
 6   Lot_Shape        2930 non-null   object 
 7   Land_Contour     2930 non-null   object 
 8   Utilities        2930 non-null   object 
 9   Lot_Config       2930 non-null   object 
 10  Land_Slope       2930 non-null   object 
 11  Neighborhood     2930 non-null   object 
 12  Condition_1      2930 non-null   object 
 13  Condition_2      2930 non-null   object 
 14  Bldg_Type        2930 non-null   object 
 15  House_Style      2930 non-null   object 
 16  Overall_Qual     2930 non-null   int64  
 17  Overall_Cond  

##2. Features classification

**1. Numerical features**

In [271]:
# define numerical features
numerical_features_all = df.select_dtypes(include=[np.number]).columns.tolist()

In [272]:
len(numerical_features_all)

37

In [273]:
len(df.select_dtypes(include=["float64", "int64"]).columns.to_list())

37

In [274]:
# Discrete numerical features
discrete_num_features = [
    'Bsmt_Full_Bath',
    'Bsmt_Half_Bath',
    'Full_Bath',
    'Half_Bath',
    'Bedroom_AbvGr',
    'Kitchen_AbvGr',
    'TotRms_AbvGrd',
    'Fireplaces',
    'Garage_Cars'
]

In [275]:
# Continuous numerical features
continuous_num_features = [
    'Lot_Frontage',
    'Lot_Area',
    'Mas_Vnr_Area',
    'BsmtFin_SF_1',
    'BsmtFin_SF_2',
    'Bsmt_Unf_SF',
    'Total_Bsmt_SF',
    '1st_Flr_SF',
    '2nd_Flr_SF',
    'Low_Qual_Fin_SF',
    'Gr_Liv_Area',
    'Garage_Area',
    'Wood_Deck_SF',
    'Open_Porch_SF',
    'Enclosed_Porch',
    '3Ssn_Porch',
    'Screen_Porch',
    'Pool_Area',
    'Misc_Val'
]

In [276]:
num_features_only = discrete_num_features + continuous_num_features
print(f'{len(num_features_only)} numerical features')

28 numerical features


**2. Categorical features**

In [277]:
# Nominal categorical features
nominal_cat_features = [
    'MS_SubClass',
    'MS_Zoning',
    'Street',
    'Alley',
    'Land_Contour',
    'Utilities',
    'Lot_Config',
    'Neighborhood',
    'Condition_1',
    'Condition_2',
    'Bldg_Type',
    'House_Style',
    'Roof_Style',
    'Roof_Matl',
    'Exterior_1st',
    'Exterior_2nd',
    'Mas_Vnr_Type',
    'Foundation',
    'Heating',
    'Central_Air',
    'Electrical',
    'Garage_Type',
    'Paved_Drive',
    'Fence',
    'Misc_Feature',
    'Sale_Type',
    'Sale_Condition'
]

In [278]:
# Ordinal categorical features
ordinal_cat_features = [
    'Lot_Shape',
    'Land_Slope',
    'Overall_Qual',
    'Overall_Cond',
    'Exter_Qual',
    'Exter_Cond',
    'Bsmt_Qual',
    'Bsmt_Cond',
    'Bsmt_Exposure',
    'BsmtFin_Type_1',
    'BsmtFin_Type_2',
    'Heating_QC',
    'Kitchen_Qual',
    'Functional',
    'Fireplace_Qu',
    'Garage_Finish',
    'Garage_Qual',
    'Garage_Cond',
    'Pool_QC'
]

In [279]:
categorical_features = nominal_cat_features + ordinal_cat_features
print(f'{len(categorical_features)} categorical featurese')

46 categorical featurese


In [280]:
set(categorical_features).intersection(set(discrete_num_features))

set()

**Change data types of `categorical_features` to `category` for memory efficiency and better performance in modeling.**

In [281]:
for col in categorical_features:
    df[col] = df[col].astype('category')

In [282]:
df[categorical_features].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 46 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   MS_SubClass     2930 non-null   category
 1   MS_Zoning       2930 non-null   category
 2   Street          2930 non-null   category
 3   Alley           198 non-null    category
 4   Land_Contour    2930 non-null   category
 5   Utilities       2930 non-null   category
 6   Lot_Config      2930 non-null   category
 7   Neighborhood    2930 non-null   category
 8   Condition_1     2930 non-null   category
 9   Condition_2     2930 non-null   category
 10  Bldg_Type       2930 non-null   category
 11  House_Style     2930 non-null   category
 12  Roof_Style      2930 non-null   category
 13  Roof_Matl       2930 non-null   category
 14  Exterior_1st    2930 non-null   category
 15  Exterior_2nd    2930 non-null   category
 16  Mas_Vnr_Type    1155 non-null   category
 17  Foundation    

**3. Temporal features**

In [283]:
date_year = ['Year_Built', 'Year_Remod/Add', 'Garage_Yr_Blt', 'Yr_Sold', ]
date_month = ['Mo_Sold']

for col in date_year:
    df[col] = pd.to_datetime(df[col], format='%Y').dt.year

for col in date_month:
    df[col] = pd.to_datetime(df[col], format='%m').dt.month

In [284]:
# Temporal features
temporal_features = date_year + date_month
temporal_features

['Year_Built', 'Year_Remod/Add', 'Garage_Yr_Blt', 'Yr_Sold', 'Mo_Sold']

**4. Target feature**

In [285]:
target = ['SalePrice']

**All features**

In [286]:
all_features = discrete_num_features + continuous_num_features + nominal_cat_features + ordinal_cat_features + temporal_features + target

In [287]:
len(all_features)

80

In [288]:
len(df.columns.to_list())

80

**Summarize**

In [289]:
print(len(discrete_num_features))
print(len(continuous_num_features))
print(len(nominal_cat_features))
print(len(ordinal_cat_features))
print(len(temporal_features))
print(len(target))

9
19
27
19
5
1


**Total features = 80**
- **Total numerical features = $37$ ($11$ discrete, $19$ continuous, $5$ temporal, $1$ target, $1$ categorical)**
- discrete numerical features = $9$
- continuous features = $37 - 11 - 5 - 1 - 1 = 19$
- temporal features = $5$
- target feature = $1$
- **total categorical features = $46$ ($2$ from discrete)**
- ordinal categorical features = $19$
- nominal categorial features = $27$

-> **Total features $= 9 + 19 + 5 + 1 + 19 + 27=80$**

In [290]:

# print columns
print(f'{len(discrete_num_features)} discrete numerical features: {discrete_num_features}')
print(f'\n{len(continuous_num_features)} continuous numerical features: {continuous_num_features}')
print(f'\n{len(ordinal_cat_features)} ordinal categorical features: {ordinal_cat_features}')
print(f'\n{len(nominal_cat_features)} nominal categorical features: {nominal_cat_features}')
print(f'\n{len(temporal_features)} temporal features: {temporal_features}')
print(f'\n{len(target)} target feature: {target}')

9 discrete numerical features: ['Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Full_Bath', 'Half_Bath', 'Bedroom_AbvGr', 'Kitchen_AbvGr', 'TotRms_AbvGrd', 'Fireplaces', 'Garage_Cars']

19 continuous numerical features: ['Lot_Frontage', 'Lot_Area', 'Mas_Vnr_Area', 'BsmtFin_SF_1', 'BsmtFin_SF_2', 'Bsmt_Unf_SF', 'Total_Bsmt_SF', '1st_Flr_SF', '2nd_Flr_SF', 'Low_Qual_Fin_SF', 'Gr_Liv_Area', 'Garage_Area', 'Wood_Deck_SF', 'Open_Porch_SF', 'Enclosed_Porch', '3Ssn_Porch', 'Screen_Porch', 'Pool_Area', 'Misc_Val']

19 ordinal categorical features: ['Lot_Shape', 'Land_Slope', 'Overall_Qual', 'Overall_Cond', 'Exter_Qual', 'Exter_Cond', 'Bsmt_Qual', 'Bsmt_Cond', 'Bsmt_Exposure', 'BsmtFin_Type_1', 'BsmtFin_Type_2', 'Heating_QC', 'Kitchen_Qual', 'Functional', 'Fireplace_Qu', 'Garage_Finish', 'Garage_Qual', 'Garage_Cond', 'Pool_QC']

27 nominal categorical features: ['MS_SubClass', 'MS_Zoning', 'Street', 'Alley', 'Land_Contour', 'Utilities', 'Lot_Config', 'Neighborhood', 'Condition_1', 'Condition_2', 'Bldg_Typ

In [291]:
# total
print(f'Numer of all features: {len(discrete_num_features) + len(continuous_num_features) + len(ordinal_cat_features) + len(nominal_cat_features) + len(temporal_features) + len(target)}')

Numer of all features: 80


## 4. Train-Test split

In [292]:
X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [293]:
X_train.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Screen_Porch,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition
381,20,RL,80.0,10400,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,MnPrv,NaN,0,6,2009,WD,Family
834,60,RL,NaN,28698,Pave,NaN,IR2,Low,AllPub,CulDSac,...,225,0,NaN,NaN,NaN,0,6,2009,WD,Abnorml
1898,20,RL,75.0,7388,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,0,NaN,NaN,NaN,0,7,2007,WD,Normal
678,90,RL,60.0,8544,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,0,NaN,NaN,NaN,0,6,2009,WD,Normal
700,70,RM,57.0,6406,Pave,Grvl,Reg,Lvl,AllPub,Inside,...,182,0,NaN,MnPrv,NaN,0,10,2009,WD,Normal


In [294]:
X_test.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Screen_Porch,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition
1357,50,RL,60.0,9144,Pave,Pave,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,3,2008,WD,Normal
2367,160,RM,21.0,1953,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,6,2006,WD,Normal
2822,20,RL,73.0,16133,Pave,NaN,Reg,HLS,AllPub,Inside,...,112,0,NaN,NaN,NaN,0,12,2006,WD,Abnorml
2126,20,RL,70.0,16561,Pave,NaN,IR2,Low,AllPub,Inside,...,0,0,NaN,GdPrv,NaN,0,7,2007,WD,Normal
1544,160,RM,24.0,1950,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,GdPrv,NaN,0,7,2008,COD,Normal


In [295]:
y_train.head()

,SalePrice
381,152000
834,185000
1898,150750
678,92900
700,150000


In [296]:
y_test.head()

,SalePrice
1357,162500
2367,83000
2822,119900
2126,147900
1544,151000


## 5. Missing Value Handling (Imputation)

In [297]:
count = X.isnull().sum().sort_values(ascending=False)
percentage = ((X.isnull().sum() / df.isnull().count()) * 100).sort_values(ascending=False)
null_values = pd.concat([count, percentage], axis=1, keys=['Count', 'Percentage'])

In [298]:
null_values.head()

,Count,Percentage
Pool_QC,2917.0,99.556314
Misc_Feature,2824.0,96.382253
Alley,2732.0,93.242321
Fence,2358.0,80.477816
Mas_Vnr_Type,1775.0,60.580205


**Visualize missing values**

In [299]:
fig = px.bar(x=null_values.index, y=null_values['Percentage'],
             labels={'x': 'Features', 'y': 'Percent of missing values'},
             title='Percent missing data by feature')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

**1. Structural Absence (Replace `NaN` with `None` or `0`)**

These columns use missing values to denote that the feature does not exist on the property.

- **`structural_categorical` (Fill with `None`)**

In [300]:
structural_categorical = [
    'Alley',

    'Bsmt_Qual',
    'Bsmt_Cond',
    'Bsmt_Exposure',
    'BsmtFin_Type_1',
    'BsmtFin_Type_2',

    'Fireplace_Qu',

    'Garage_Type',
    'Garage_Finish',
    'Garage_Qual',
    'Garage_Cond',

    'Pool_QC',
    'Fence',
    'Misc_Feature'
]

```
df.Alley=["No Alley" if x is np.nan else x for x in df.Alley]
df.Mas_Vnr_Type=["Missing" if x is np.nan else x for x in df.Mas_Vnr_Type]
df.Bsmt_Qual=["No Bsmnt" if x is np.nan else x for x in df.Bsmt_Qual]
df.Bsmt_Cond=["No Bsmnt" if x is np.nan else x for x in df.Bsmt_Cond]
df.Bsmt_Exposure=["No Bsmnt" if x is np.nan else x for x in df.Bsmt_Exposure]
df.BsmtFin_Type_1=["No Bsmnt" if x is np.nan else x for x in df.BsmtFin_Type_1]
df.BsmtFin_Type_2=["No Bsmnt" if x is np.nan else x for x in df.BsmtFin_Type_2]
df.Fireplace_Qu=["No Firepl" if x is np.nan else x for x in df.Fireplace_Qu]
df.Garage_Type=["No Grg" if x is np.nan else x for x in df.Garage_Type]
df.Garage_Finish=["No Grg" if x is np.nan else x for x in df.Garage_Finish]
df.Garage_Qual=["No Grg" if x is np.nan else x for x in df.Garage_Qual]
df.Garage_Cond=["No Grg" if x is np.nan else x for x in df.Garage_Cond]
df.Pool_QC=["No Pool" if x is np.nan else x for x in df.Pool_QC]
df.Fence=["No Fence" if x is np.nan else x for x in df.Fence]
df.Misc_Feature=["None" if x is np.nan else x for x in df.Misc_Feature]
```

```py
df['Alley'] = df['Alley'].fillna('No Alley')
df['Mas_Vnr_Type'] = df['Mas_Vnr_Type'].fillna('Missing')
df['Bsmt_Qual'] = df['Bsmt_Qual'].fillna('No Bsmnt')
df['Bsmt_Cond'] = df['Bsmt_Cond'].fillna('No Bsmnt')
df['Bsmt_Exposure'] = df['Bsmt_Exposure'].fillna('No Bsmnt')
df['BsmtFin_Type_1'] = df['BsmtFin_Type_1'].fillna('No Bsmnt')
df['BsmtFin_Type_2'] = df['BsmtFin_Type_2'].fillna('No Bsmnt')
df['Fireplace_Qu'] = df['Fireplace_Qu'].fillna('No Firepl')
df['Garage_Type'] = df['Garage_Type'].fillna('No Grg')
df['Garage_Finish'] = df['Garage_Finish'].fillna('No Grg')
df['Garage_Qual'] = df['Garage_Qual'].fillna('No Grg')
df['Garage_Cond'] = df['Garage_Cond'].fillna('No Grg')
df['Pool_QC'] = df['Pool_QC'].fillna('No Pool')
df['Fence'] = df['Fence'].fillna('No Fence')
df['Misc_Feature'] = df['Misc_Feature'].fillna('None')
```

In [301]:
fill_values = {
    'Alley': 'No Alley Access',
    'Bsmt_Qual': 'No Basement',
    'Bsmt_Cond': 'No Basement',
    'Bsmt_Exposure': 'No Basement',
    'BsmtFin_Type_1': 'No Basement',
    'BsmtFin_Type_2': 'No Basement',
    'Fireplace_Qu': 'No Fireplace',
    'Garage_Type': 'No Garage',
    'Garage_Finish': 'No Garage',
    'Garage_Qual': 'No Garage',
    'Garage_Cond': 'No Garage',
    'Pool_QC': 'No Pool',
    'Fence': 'No Fence',
    'Misc_Feature': 'None'
}

for col, fill_value in fill_values.items():
    if fill_value not in X[col].cat.categories:
        X[col] = X[col].cat.add_categories(fill_value)
    X[col] = X[col].fillna(fill_value)

In [302]:
X[structural_categorical].isnull().sum()

,0
Alley,0
Bsmt_Qual,0
Bsmt_Cond,0
Bsmt_Exposure,0
BsmtFin_Type_1,0
BsmtFin_Type_2,0
Fireplace_Qu,0
Garage_Type,0
Garage_Finish,0
Garage_Qual,0


- **`structural_numerical` (Fill with `0`)**

In [303]:
structural_numerical = [
    'BsmtFin_SF_1', 'BsmtFin_SF_2', 'Bsmt_Unf_SF', 'Total_Bsmt_SF',
    'Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Garage_Cars', 'Garage_Area', 'Mas_Vnr_Area'
]

In [304]:
for col in structural_numerical:
   X[col] = X[col].fillna(0)

In [305]:
X[structural_numerical].isnull().sum()

,0
BsmtFin_SF_1,0
BsmtFin_SF_2,0
Bsmt_Unf_SF,0
Total_Bsmt_SF,0
Bsmt_Full_Bath,0
Bsmt_Half_Bath,0
Garage_Cars,0
Garage_Area,0
Mas_Vnr_Area,0


**Checking**

In [306]:
X[structural_numerical + structural_categorical].isnull().sum().sum()

np.int64(0)

In [307]:
len(structural_numerical + structural_categorical)

23

In [308]:
X[structural_categorical + structural_numerical].head()

,Alley,Bsmt_Qual,Bsmt_Cond,Bsmt_Exposure,BsmtFin_Type_1,BsmtFin_Type_2,Fireplace_Qu,Garage_Type,Garage_Finish,Garage_Qual,...,Misc_Feature,BsmtFin_SF_1,BsmtFin_SF_2,Bsmt_Unf_SF,Total_Bsmt_SF,Bsmt_Full_Bath,Bsmt_Half_Bath,Garage_Cars,Garage_Area,Mas_Vnr_Area
0,No Alley Access,TA,Gd,Gd,BLQ,Unf,Gd,Attchd,Fin,TA,...,None,639.0,0.0,441.0,1080.0,1.0,0.0,2.0,528.0,112.0
1,No Alley Access,TA,TA,No,Rec,LwQ,No Fireplace,Attchd,Unf,TA,...,None,468.0,144.0,270.0,882.0,0.0,0.0,1.0,730.0,0.0
2,No Alley Access,TA,TA,No,ALQ,Unf,No Fireplace,Attchd,Unf,TA,...,Gar2,923.0,0.0,406.0,1329.0,0.0,0.0,1.0,312.0,108.0
3,No Alley Access,TA,TA,No,ALQ,Unf,TA,Attchd,Fin,TA,...,None,1065.0,0.0,1045.0,2110.0,1.0,0.0,2.0,522.0,0.0
4,No Alley Access,Gd,TA,No,GLQ,Unf,TA,Attchd,Fin,TA,...,None,791.0,0.0,137.0,928.0,0.0,0.0,2.0,482.0,0.0


In [309]:
features_with_missing_values = list(set(X.columns.to_list()) - set(structural_numerical + structural_categorical))
print(len(features_with_missing_values))
features_with_missing_values

56


['Lot_Shape',
 'Exter_Qual',
 'Gr_Liv_Area',
 'Fireplaces',
 'Lot_Frontage',
 'Overall_Qual',
 'Roof_Matl',
 'Exterior_2nd',
 'Utilities',
 '3Ssn_Porch',
 'Sale_Condition',
 'Wood_Deck_SF',
 'Exterior_1st',
 'Misc_Val',
 'Functional',
 'Half_Bath',
 'Foundation',
 'Year_Built',
 'Mo_Sold',
 'Central_Air',
 'Paved_Drive',
 'TotRms_AbvGrd',
 'Yr_Sold',
 'Condition_2',
 'Low_Qual_Fin_SF',
 'Heating_QC',
 'Lot_Config',
 'Lot_Area',
 'Open_Porch_SF',
 'Mas_Vnr_Type',
 'Sale_Type',
 'Kitchen_Qual',
 'Bedroom_AbvGr',
 '1st_Flr_SF',
 'Year_Remod/Add',
 'Roof_Style',
 'Neighborhood',
 'Exter_Cond',
 'Condition_1',
 'Full_Bath',
 'Land_Contour',
 'House_Style',
 'Land_Slope',
 'Bldg_Type',
 'Street',
 'Overall_Cond',
 'Enclosed_Porch',
 'Kitchen_AbvGr',
 'MS_SubClass',
 '2nd_Flr_SF',
 'Electrical',
 'Garage_Yr_Blt',
 'Pool_Area',
 'Heating',
 'Screen_Porch',
 'MS_Zoning']

> Out of $80$ total features, $22$ features with missing values are handled considering _structuring absence_. Remaining $58$ will be handled by _statistical imputation_ and other ways.

In [310]:
df.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition,SalePrice
0,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


**2. Statistical Imputation (True Missingness/Clerical Errors)**

These features are physically required for a house to exist. Missing data here represents a processing omission or clerical typo.

- **`spatial_numerical` (Impute using the median of their specific `Neighborhood`)**

In [311]:
spatial_numerical = ['Lot_Frontage']

In [312]:
X[spatial_numerical].isnull().sum()

,0
Lot_Frontage,490


In [313]:
X['Lot_Frontage'].interpolate(method='linear', axis=0, inplace=True)

In [314]:
X[spatial_numerical].isnull().sum()

,0
Lot_Frontage,0


- **`mode_categorical` (Impute using the column's global mode/most frequent item)**

In [315]:
mode_categorical = [
    'Utilities', 'Exterior_1st', 'Exterior_2nd', 'Electrical',
    'Kitchen_Qual', 'Functional', 'Sale_Type'
]

In [316]:
X[mode_categorical].isnull().sum()

,0
Utilities,0
Exterior_1st,0
Exterior_2nd,0
Electrical,1
Kitchen_Qual,0
Functional,0
Sale_Type,0


In [317]:
X[mode_categorical].head()

,Utilities,Exterior_1st,Exterior_2nd,Electrical,Kitchen_Qual,Functional,Sale_Type
0,AllPub,BrkFace,Plywood,SBrkr,TA,Typ,WD
1,AllPub,VinylSd,VinylSd,SBrkr,TA,Typ,WD
2,AllPub,Wd Sdng,Wd Sdng,SBrkr,Gd,Typ,WD
3,AllPub,BrkFace,BrkFace,SBrkr,Ex,Typ,WD
4,AllPub,VinylSd,VinylSd,SBrkr,TA,Typ,WD


In [318]:
for col in mode_categorical:
    if df[col].isnull().sum() > 0:
        X[col] = X[col].fillna(X[col].mode()[0])

- **`special_case_temporal` (Impute missing indices dynamically with that row's `Year_Built` value)**

In [319]:
special_case_temporal = ['Garage_Yr_Blt']

In [320]:
X[temporal_features].isnull().sum()

,0
Year_Built,0
Year_Remod/Add,0
Garage_Yr_Blt,159
Yr_Sold,0
Mo_Sold,0


In [321]:
X[temporal_features].head()

,Year_Built,Year_Remod/Add,Garage_Yr_Blt,Yr_Sold,Mo_Sold
0,1960,1960,1960.0,2010,5
1,1961,1961,1961.0,2010,6
2,1958,1958,1958.0,2010,6
3,1968,1968,1968.0,2010,4
4,1997,1998,1997.0,2010,3


In [322]:
X['Garage_Yr_Blt'] = X['Garage_Yr_Blt'].fillna(X['Year_Built'])

In [323]:
X[temporal_features].isnull().sum()

,0
Year_Built,0
Year_Remod/Add,0
Garage_Yr_Blt,0
Yr_Sold,0
Mo_Sold,0


The data dictionary describes `MasVnrType` as masonry veneer type and `MasVnrArea` as veneer area.

In [324]:
X[['Mas_Vnr_Area', 'Mas_Vnr_Area']].isnull().sum()

,0
Mas_Vnr_Area,0
Mas_Vnr_Area,0


In [325]:
X.loc[
    X["Mas_Vnr_Type"].eq("None") &
    X["Mas_Vnr_Area"].isna(),
    "Mas_Vnr_Area"
] = 0

In [326]:
X['Mas_Vnr_Type'] = X['Garage_Yr_Blt'].fillna(X['Year_Built'])

**Final check for existence of null values**

In [327]:
# Null values left?
print(X.isnull().values.any())
if X.isnull().values.any():
  print("Yes, there exist null values!")
  null_counts = X.isnull().sum()
  null_counts = null_counts[null_counts > 0]
  print(null_counts)
else:
  print("No null values")

False
No null values


> **Final dataset with no missing values**

In [336]:
X[['Mas_Vnr_Area', 'Mas_Vnr_Type']].head()

,Mas_Vnr_Area,Mas_Vnr_Type
0,112.0,1960.0
1,0.0,1961.0
2,108.0,1958.0
3,0.0,1968.0
4,0.0,1997.0


In [337]:
X.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Screen_Porch,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition
0,20,RL,141.0,31770,Pave,No Alley Access,IR1,Lvl,AllPub,Corner,...,0,0,No Pool,No Fence,None,0,5,2010,WD,Normal
1,20,RH,80.0,11622,Pave,No Alley Access,Reg,Lvl,AllPub,Inside,...,120,0,No Pool,MnPrv,None,0,6,2010,WD,Normal
2,20,RL,81.0,14267,Pave,No Alley Access,IR1,Lvl,AllPub,Corner,...,0,0,No Pool,No Fence,Gar2,12500,6,2010,WD,Normal
3,20,RL,93.0,11160,Pave,No Alley Access,Reg,Lvl,AllPub,Corner,...,0,0,No Pool,No Fence,None,0,4,2010,WD,Normal
4,60,RL,74.0,13830,Pave,No Alley Access,IR1,Lvl,AllPub,Inside,...,0,0,No Pool,MnPrv,None,0,3,2010,WD,Normal


In [338]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 79 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   MS_SubClass      2930 non-null   category
 1   MS_Zoning        2930 non-null   category
 2   Lot_Frontage     2930 non-null   float64 
 3   Lot_Area         2930 non-null   int64   
 4   Street           2930 non-null   category
 5   Alley            2930 non-null   category
 6   Lot_Shape        2930 non-null   category
 7   Land_Contour     2930 non-null   category
 8   Utilities        2930 non-null   category
 9   Lot_Config       2930 non-null   category
 10  Land_Slope       2930 non-null   category
 11  Neighborhood     2930 non-null   category
 12  Condition_1      2930 non-null   category
 13  Condition_2      2930 non-null   category
 14  Bldg_Type        2930 non-null   category
 15  House_Style      2930 non-null   category
 16  Overall_Qual     2930 non-null   category


## 6. Outlier Removal/Resolving

Important concepts in outlier detection! Here's a breakdown:

*   **Interquartile Range (IQR):** The IQR is a measure of statistical dispersion, representing the range of the middle 50% of your data. It's calculated as the difference between the third quartile (Q3) and the first quartile (Q1).

    *   **Q1 (First Quartile):** The value below which 25% of the data falls.
    *   **Q3 (Third Quartile):** The value below which 75% of the data falls.
    *   `IQR = Q3 - Q1`

*   **Outlier Thresholds (Lower Bound and Upper Bound):** These are calculated boundaries used to identify potential outliers in a dataset. They are derived using the IQR.

    *   **Lower Bound:** `Q1 - 1.5 * IQR`
    *   **Upper Bound:** `Q3 + 1.5 * IQR`

*   **Outlier vs. IQR:**
    *   The **IQR** itself is a measure of the spread of the central portion of the data, robust to extreme values.
    *   **Outliers** are data points that fall outside the calculated lower and upper bounds. The 1.5 multiplier in the threshold calculation is a commonly used convention (often referred to as the "1.5 IQR rule") to define what constitutes an outlier relative to the bulk of the data's spread. Any data point less than the lower bound or greater than the upper bound is considered an outlier.

In [333]:
num_features_only

['Bsmt_Full_Bath',
 'Bsmt_Half_Bath',
 'Full_Bath',
 'Half_Bath',
 'Bedroom_AbvGr',
 'Kitchen_AbvGr',
 'TotRms_AbvGrd',
 'Fireplaces',
 'Garage_Cars',
 'Lot_Frontage',
 'Lot_Area',
 'Mas_Vnr_Area',
 'BsmtFin_SF_1',
 'BsmtFin_SF_2',
 'Bsmt_Unf_SF',
 'Total_Bsmt_SF',
 '1st_Flr_SF',
 '2nd_Flr_SF',
 'Low_Qual_Fin_SF',
 'Gr_Liv_Area',
 'Garage_Area',
 'Wood_Deck_SF',
 'Open_Porch_SF',
 'Enclosed_Porch',
 '3Ssn_Porch',
 'Screen_Porch',
 'Pool_Area',
 'Misc_Val']

In [339]:
def outlier_thresholds(X, column):
    Q1 = X[column].quantile(0.25)
    Q3 = X[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    #outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return lower_bound, upper_bound

In [335]:
# detect outliers
def detect_outliers(X, column):
    if pd.api.types.is_numeric_dtype(df[column]):
        low, up = outlier_thresholds(df, column)
        return (df[column] > up) | (df[column] < low)
    else:
        return pd.Series(False, index=df.index)

In [ ]:
outliers_df = pd.DataFrame(index=df.index)
for col in numerical_features_all:
    outliers_df[col] = detect_outliers(df, col)

# Display the count of outliers per column
print("Number of outliers per numerical column:")
print(outliers_df.sum())

In [ ]:
len(numerical_features_all)

In [ ]:
len(discrete_num_features)

In [ ]:
len(num_feats_int)

> $26 = 37 - 12 + 1$

In [ ]:
display(outliers_df.head())

In [ ]:
# Function to check all numeric columns for outliers
def inspect_outliers(df, col_list):
    results = {}
    for col in col_list:
        results[col] = detect_outliers(df, col).any() if pd.api.types.is_numeric_dtype(df[col]) else False
    return results

In [ ]:
# Function to display columns with outlier information
def display_outlier_info(df, col_list):
    print("Outlier thresholds for the numeric column")
    for col in col_list:
        low, up = outlier_thresholds(df, col)
        print(f"{col} : Lower Bound: {low}, Upper Bound: {up}", end="\n")
    print(end="\n\n")
    outliers_absent = []
    outliers_present = []
    for col, value in inspect_outliers(df, col_list).items():
        if value:
            outliers_present.append(col)
        else:
            outliers_absent.append(col)
    print("Columns with outliers")
    print(outliers_present)
    print(f"column count: {len(outliers_present)}", end="\n\n")
    print("Columns with no outliers")
    print(outliers_absent)
    print(f"column count: {len(outliers_absent)}", end="\n\n")

    return outliers_present, outliers_absent

In [ ]:
# Resolve Outliers
def threshold_resolution(df, column):
    low, up = outlier_thresholds(df, column)
    df.loc[df[column] < low, column] = low
    df.loc[df[column] > up, column] = up
    return low, up

In [ ]:
current_numerical_columns = df.select_dtypes(include=[np.number]).columns.tolist()
outliers_present, outliers_absent = display_outlier_info(df, current_numerical_columns)

**Now remove outliers**

In [ ]:
for col in outliers_present:
    if col not in ['SalePrice']:
        threshold_resolution(df, col)

In [ ]:
outliers_present, outliers_absent = display_outlier_info(df, current_numerical_columns)

> **Now no columns with outliers except `SalePrice`**

## 7. Feature Creation / Engineering

In [ ]:
import numpy as np
# New Features for Housing Dataset

# Total square footage
df['TotalSF'] = df['1st_Flr_SF'] + df['2nd_Flr_SF'] + df['Total_Bsmt_SF']

# Age of the house at time of sale
df['Age'] = df['Yr_Sold'] - df['Year_Built']

# Ensure Half_Bath and Bsmt_Half_Bath are numeric for calculations
df['Half_Bath'] = df['Half_Bath'].astype(int)
df['Bsmt_Half_Bath'] = df['Bsmt_Half_Bath'].astype(int)
df['Full_Bath'] = df['Full_Bath'].astype(int)
df['Bsmt_Full_Bath'] = df['Bsmt_Full_Bath'].astype(int)

# Total number of bathrooms
df['TotalBathrooms'] = (df['Full_Bath'] + (0.5 * df['Half_Bath']) +
                        df['Bsmt_Full_Bath'] + (0.5 * df['Bsmt_Half_Bath'])).astype('category')

# Boolean indicator for pool presence
df['HasPool'] = df['Pool_Area'].apply(lambda x: 1 if x > 0 else 0).astype('category')

# Total porch area
df['TotalPorchSF'] = df['Open_Porch_SF'] + df['Enclosed_Porch'] + df['3Ssn_Porch'] + df['Screen_Porch']

# Ensure Garage_Cars is numeric for calculations
df['Garage_Cars'] = df['Garage_Cars'].astype(int)

# Total parking capacity, considering paved drive
df['TotalParking'] = (df['Garage_Cars'] + df['Paved_Drive'].apply(lambda x:1 if x == 'Y' else 0)).astype('category')
# Boolean indicator for remodeling
df['Remodeled'] = (df['Year_Remod/Add'] != df['Year_Built']).astype(int).astype('category')
# Years since last remodel at time of sale
df['YearsSinceRemodel'] = df['Yr_Sold'] - df['Year_Remod/Add'].astype(int)

# Total number of rooms
df['TotalRooms'] = df['TotRms_AbvGrd'].astype('category') # TotRms_AbvGrd is already category

# Ratio of living area to lot area
df['LotRatio'] = df['Gr_Liv_Area'] / df['Lot_Area']

# Sale price per square foot (if SalePrice is available)
if 'SalePrice' in df.columns:
    df['PricePerSqFt'] = df['SalePrice'] / df['TotalSF']

# Categorical age group of the house
def age_group(age):
    if age <= 10:
        return 'New'
    elif age <= 30:
        return 'Modern'
    elif age <= 50:
        return 'Mature'
    else:
        return 'Old'

df['AgeGroup'] = df['Age'].apply(age_group).astype('category')

# Total outdoor living space
df['TotalOutdoorSF'] = df['Wood_Deck_SF'] + df['TotalPorchSF']

# Boolean indicator for fireplace presence
df['HasFireplace'] = df['Fireplaces'].apply(lambda x: 1 if x > 0 else 0).astype('category')

In [ ]:
# Mapping for OverallQual to match ExterQual and KitchenQual ratings
overall_qual_map = {
    10: 'Ex',  # Very Excellent
    9:  'Ex',  # Excellent
    8:  'Gd',  # Very Good
    7:  'Gd',  # Good
    6:  'TA',  # Above Average
    5:  'TA',  # Average
    4:  'Fa',  # Below Average
    3:  'Fa',  # Fair
    2:  'Po',  # Poor
    1:  'Po'   # Very Poor
}

# Apply the mapping to convert OverallQual to qualitative categories
df['OverallQualCategory'] = df['Overall_Qual'].map(overall_qual_map).astype('category')

# Print the updated dataframe to check the new category column
print(df[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual', 'Kitchen_Qual']].head())


In [ ]:
df[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual','Kitchen_Qual']].dtypes

In [ ]:
categorical_features_new = ['AgeGroup', 'HasPool','Remodeled', 'HasFireplace', 'TotalRooms', 'TotalParking', 'TotalBathrooms', 'OverallQualCategory']

**Change data types of newly created `categorical_features` to `category` for memory efficiency and better performance in modeling.**

In [ ]:
# for col in categorical_features_new:
#     df[col] = df[col].astype('category')

In [ ]:
# Mapping for ExterQual and KitchenQual
quality_map = {
    'Ex': 5,  # Excellent
    'Gd': 4,  # Good
    'TA': 3,  # Typical/Average
    'Fa': 2,  # Fair
    'Po': 1   # Poor
}

df['Exter_Qual'] = df['Exter_Qual'].map(quality_map)
df['Kitchen_Qual'] = df['Kitchen_Qual'].map(quality_map)
df['OverallQualCategory'] = df['OverallQualCategory'].map(quality_map)

In [ ]:
print(df[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual', 'Kitchen_Qual']].head())

In [ ]:
print(df[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual', 'Kitchen_Qual']].dtypes)

In [ ]:
len(df.columns.to_list())

In [ ]:
df.head()

In [ ]:
# Combined quality score
df['Quality_Score'] = ((df['OverallQualCategory'].astype(int) + df['Exter_Qual'].astype(int) + df['Kitchen_Qual'].astype(int)) / 3)

In [ ]:
df.columns.to_list()

In [ ]:
categorical_features_new

In [ ]:
df[categorical_features_new].dtypes

In [ ]:
len(df.columns.to_list())

## 8. Mathematical Transformation

## 9. Categorical Encoding

In [ ]:
categorical_features_new

In [ ]:
categorical_features_old = categorical_features_all

In [ ]:
categorical_features_final = categorical_features_old + categorical_features_new

In [ ]:
print(len(categorical_features_old))
print(len(categorical_features_new))
print(len(categorical_features_final))

In [ ]:
categorical_features_final

In [ ]:
for col in categorical_features_final:
  print(f"{col}: {df[col].nunique()}")

In [ ]:
from sklearn.model_selection import train_test_split, LeaveOneOut, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, scale, StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [ ]:
binary_columns = [col for col in df.columns if (df[col].dtypes not in ["float64","int64","int32"]) & (df[col].nunique() == 2)]
binary_columns

In [ ]:
labelencoder = LabelEncoder()

for col in binary_columns:
    df[col] = labelencoder.fit_transform(df[col])

In [ ]:
ohe_cols = [col for col in df.columns if (df[col].dtype not in ['float64','int64','int32']) & ((df[col].nunique() > 2) | (df[col].nunique() == 1))]
ohe_cols

In [ ]:
df = pd.get_dummies(df, columns=ohe_cols).reset_index(drop=True)
df.shape

In [ ]:
df = df.loc[:,~df.columns.duplicated()]

In [ ]:
len(df.columns.to_list())

> From $96$ to $427$ columns after encoding

## 10. Scaling

In [ ]:
scaling_features=[feature for feature in df.columns if feature not in ['SalePrice'] ]
len(scaling_features)

In [ ]:
df[scaling_features].head()

In [ ]:
minmax_scaler = MinMaxScaler()
df[scaling_features] = minmax_scaler.fit_transform(df[scaling_features])

In [ ]:
numerical_features_all

In [ ]:
set(temporal_features) - set(numerical_features_all)

In [ ]:
len(numerical_features_all)

In [ ]:
numerical_features_new = list(set())

In [ ]:
df['SalePrice']

In [ ]:
df.head()

## 11. `Pipeline` and `ColumnTransformer`

While a `ColumnTransformer` is typically used earlier in the process to apply different transformations to different subsets of columns (e.g., numerical vs. categorical), our data (`df`) has already undergone comprehensive preprocessing, including handling missing values, scaling numerical features, and one-hot encoding categorical features.

At this stage, all features in our `df` are numerical and scaled, making a `ColumnTransformer` redundant for *further* preprocessing steps. However, a `Pipeline` is still invaluable for chaining model steps and ensuring consistency.

I will now re-split our preprocessed `df` into `X` (features) and `y` (target variable `SalePrice`), then define a simple `Pipeline` that includes a model, ready for training.

> On an other notebook

## 12. Modeling (Training)

### Models
**Baseline and Linear Models**
- Linear Regression (OLS)
- Ridge and Lasso Regression

**Tree-Based and Ensemble Models**
- Random Forest Regressor
- XGBoost (Extreme Gradient Boosting)
- LightGBM
- CatBoost

**Non-Linear Models**
- Support Vector Regressor (SVR)
### Evaluation Metrics
- **MAE (Mean Absolute Error)**: Measures average absolute distance between predictions and true values in original currency units.
- **RMSE (Root Mean Squared Error)**: Penalizes larger errors more heavily by squaring differences before taking the square root.
- **R² Score**: Represents the proportion of variance in house prices explained by the model features

In [ ]:
df.tail()

In [ ]:
train_df = df.iloc[:1460]
test_df = df.iloc[1460:]

In [ ]:
# Reset the index for both dataframes
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
#drop Target Feature from test_df
test_df = test_df.drop('SalePrice', axis=1)

In [ ]:
train_df.shape, test_df.shape

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
# Prepare the data
y = df['SalePrice']
X = df.drop(['SalePrice'], axis=1)

In [ ]:
# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
y_train.head()

In [ ]:
y_test.head()

**Import models**

In [ ]:
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score
from catboost import CatBoostRegressor

In [ ]:
def evaluate_metrics(true, predicted):
    mse = mean_squared_error(true, predicted) # Calculate Accuracy
    mae = mean_absolute_error(true, predicted) # Calculated MAE
    rmse = root_mean_squared_error(true, predicted) # Calculated RMSE
    r2 = r2_score(true, predicted) # Calculated R2 Score
    return mse, mae, rmse, r2

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso Regression": Lasso(),
    "Ridge Regression": Ridge(),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "CatBoosting Regressor": CatBoostRegressor(verbose=False, random_state=42),
    "AdaBoost Regressor": AdaBoostRegressor(random_state=42),
    "Support Vector Regressor": SVR()
}

In [ ]:
# Create a function which can evaluate models and return a report
def evaluate_regressor_models(X_train, y_train, X_test, y_test, models):
    '''
    This function takes in X_train, y_train, X_test, y_test and models dictionary as input
    Iterates through the given model dictionary and evaluates the metrics
    Returns: Dataframe which contains report of all models metrics with cost
    '''
    models_list = []
    mse_list = []

    for i in range(len(list(models))):
        model = list(models.values())[i]
        model.fit(X_train, y_train) # Train model

        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        # Training set performance
        mse_train, mae_train, rmse_train, r2_train = evaluate_metrics(y_train ,y_train_pred)


        # Test set performance
        mse_test, mae_test, rmse_test, r2_test = evaluate_metrics(y_test, y_test_pred)

        print(list(models.keys())[i])
        models_list.append(list(models.keys())[i])

        print('Model performance for Training set')
        print(f"- MSE: {mse_train:.4f}")
        print(f'- MAE: {mae_train:.4f}')
        print(f'- RMSE: {rmse_train:.4f}')
        print(f'- R2: {r2_train:.4f}')

        print('----------------------------------')

        print('Model performance for Test set')
        print(f'- MSE: {mse_test:.4f}')
        mse_list.append(mse_test)
        print(f'- MAE: {mae_test:.4f}')
        print(f'- RMSE: {rmse_test:.4f}')
        print(f'- R2: {r2_test:.4f}')
        print('='*35)
        print('\n')

    report=pd.DataFrame(list(zip(models_list, mse_list)), columns=['Model Name', 'MSE']).sort_values(by=['MSE'], ascending=True) # Sort by MSE ascending, lower is better

    return report

In [ ]:
base_model_report = evaluate_regressor_models(X_train, y_train, X_test, y_test, models)

**Model Performance Visualization**

In [ ]:
print('\nModel Performance Visualization')
for model_name, regressor in models.items():
    # Ensure the model is trained before predicting
    # The evaluate_regressor_models function already trains the models
    # So, we can directly predict using the models in the 'models' dictionary.
    y_pred = regressor.predict(X_test)

    plt.figure(figsize=(16, 6))

    # Regression Plot
    plt.subplot(1, 2, 1)
    sns.regplot(x=y_pred, y=y_test, scatter_kws={'alpha':0.4})
    plt.xlabel('Predicted Price')
    plt.ylabel('Actual Price')
    plt.title(f'Regression Plot: Actual vs Predicted ({model_name})')

    # Residual Plot
    plt.subplot(1, 2, 2)
    residuals = y_test - y_pred
    sns.scatterplot(x=y_pred, y=residuals, alpha=0.4)
    plt.axhline(y=0, color='r', linestyle='--')
    plt.xlabel('Predicted Price')
    plt.ylabel('Residuals')
    plt.title(f'Residual Plot ({model_name})')

    plt.tight_layout()
    plt.show()

**Cross Validation**

In [ ]:
from sklearn.model_selection import cross_validate

print("Cross-Validation Results:")
for model_name, regressor in models.items():
    scoring = {'mae': 'neg_mean_absolute_error', 'mse': 'neg_mean_squared_error', 'r2': 'r2'}
    cv_results = cross_validate(regressor, X, y, cv=5, scoring=scoring)
    print(f"\n{model_name}:")
    print(f"  Average MAE: {-cv_results['test_mae'].mean():.4f}")
    print(f"  Average MSE: {-cv_results['test_mse'].mean():.4f}")
    print(f"  Average RMSE: {np.sqrt(-cv_results['test_mse'].mean()):.4f}")
    print(f"  Average R2: {cv_results['test_r2'].mean():.4f}")

**Hyperparameter Tuning**

In [ ]:
# default parameters
print("Default Parameters:")
for model_name, regressor in models.items():
    print(f'{model_name}')
    print(f'parameters: {regressor.get_params()}', end='\n\n')

In [ ]:
# Define parameter grids for hyperparameter tuning
model_params = [
    ('Linear Regression', LinearRegression(), {
        'fit_intercept': [True, False]
    }),
    ('Lasso Regression', Lasso(), {
        'alpha': [0.1, 1.0, 10.0],
        'fit_intercept': [True, False]
    }),
    ('Ridge Regression', Ridge(), {
        'alpha': [0.1, 1.0, 10.0],
        'fit_intercept': [True, False]
    }),
    ('Random Forest', RandomForestRegressor(random_state=42), {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10]
    }),
    ('GBM', GradientBoostingRegressor(random_state=42), {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.8, 1.0],
        'max_depth': [3, 5, 7]
    })
]

In [ ]:
# Best parameters and scores initialization
best_model_name = None
best_model = None
best_score = float('-inf')
best_params = None

print("\nHyperparameter Tuning Results:")
for model_name, regressor, params in model_params:
    regressor_grid = GridSearchCV(regressor, params, cv=5, n_jobs=-1, verbose=1).fit(X, y)

    print(f"\nModel: {model_name}")
    print("Best Parameters:", regressor_grid.best_params_)
    print("Best Score:", regressor_grid.best_score_)
    print("\n")

    if regressor_grid.best_score_ > best_score:
        best_model_name = model_name
        best_model = regressor
        best_score = regressor_grid.best_score_
        best_params = regressor_grid.best_params_

print("\n\nFinal Inference")
print(f"Model: {best_model_name}")
print("Best Parameters:", best_params)
print("Best Score:", best_score)


In [ ]:
best_model

In [ ]:
# Function to find best model parameters
def best_params(model_params, best_model):
    for model_name, regressor, params in model_params.items():
        if best_model == regressor:
            return params
    return None

In [ ]:
# Apply the function
best_model_params = best_params(model_params, best_model)

In [ ]:
best_model_params

In [ ]:
# Final GridSearchCV with best model and its parameter grid
regressor_best_grid = GridSearchCV(best_model, best_model_params, cv=5, n_jobs=-1, verbose=1).fit(X, y)

In [ ]:
regressor_best_grid

In [ ]:
final_model = GradientBoostingRegressor(**regressor_best_grid.best_params_).fit(X, y)

**Cross Validation**

In [ ]:
print("Best Model Cross-Validation")
scoring = {'mae': 'neg_mean_absolute_error', 'mse': 'neg_mean_squared_error', 'r2': 'r2'}
cv_results_final = cross_validate(final_model, X, y, cv=5, scoring=scoring)
print(f"\n{model_name}:")
print(f"  Average MAE: {-cv_results['test_mae'].mean():.4f}")
print(f"  Average MSE: {-cv_results['test_mse'].mean():.4f}")
print(f"  Average RMSE: {np.sqrt(-cv_results['test_mse'].mean()):.4f}")
print(f"  Average R2: {cv_results['test_r2'].mean():.4f}", end="\n\n")

**Feature Importance**

In [ ]:
# creating function to visualize:

def visualize_importance(model, features, start=0, num=len(X), save=False):
    feature_imp = pd.DataFrame({'Value': model.feature_importances_, 'Feature': features.columns})
    sorted_feature_imp = feature_imp.sort_values("Value", ascending=False)[start:num]
    display(sorted_feature_imp)
    #print(feature_imp.sort_values("Value",ascending=False)[start:num])
    plt.figure(figsize=(12, 12))
    sns.set(font_scale=0.7)
    palette = sns.color_palette("coolwarm", n_colors=num-start)
    sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value",ascending=False)[start:num], palette=palette)
    plt.title('Visualization of Important Features')
    plt.tight_layout()
    plt.show()

In [ ]:
visualize_importance(final_model, X,start=0, num=45)

In [ ]:
predictions = final_model.predict(test_df.drop(['Id'], axis=1))

In [ ]:
predictions

In [ ]:
test_sample = test_df.drop(['Id'], axis=1).iloc[0].values.reshape(1, -1)
prediction = final_model.predict(test_sample)
print(prediction)

In [ ]:
predict_df = test_df.copy()
predict_df = predict_df.drop(['Id'], axis=1)

##13. Saving model with Pickle

In [ ]:
import pickle

with open('arko_ml_model.pkl', 'wb') as file:
    pickle.dump(final_model, file)

print("Model saved to arko_ml_model.pkl")

**Loading model and predicting using it**

In [ ]:
# Load model from pickle file
with open('arko_ml_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

result = loaded_model.score(X_test, y_test)
print(result)

print('Predicted SalePrice is : ', loaded_model.predict(predict_df))